# Store Intelligence Report

**Actionable insights, risks, and opportunities for a single store.**

This notebook queries the `lh_retaildata` Lakehouse views, applies rule-based risk/opportunity detection with dollar-value quantification, then uses Azure OpenAI to synthesize an executive narrative.

### Parameters
- `store_id` — The store ID to analyze (e.g., `1`)
- `lookback_days` — Analysis window in days (default: 7)

In [ ]:
# ── Cell 1: Parameters ────────────────────────────────────────────────
store_id = 1               # Change to target store ID
lookback_days = 7          # Analysis window (days)

In [ ]:
# ── Cell 2: Imports & Configuration ───────────────────────────────────
import json
from datetime import datetime, timedelta

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Risk thresholds (edit as needed) ──
REVENUE_CRITICAL_DROP  = 15.0    # WoW revenue decline % = critical
REVENUE_WARNING_DROP   = 5.0     # WoW revenue decline % = warning

# ── Date boundaries ──
today = datetime.now().date()
period_end   = today
period_start = today - timedelta(days=lookback_days)
prior_end    = period_start - timedelta(days=1)
prior_start  = prior_end - timedelta(days=lookback_days - 1)
osa_14d_start = today - timedelta(days=14)

print(f"Analysis period : {period_start} → {period_end}")
print(f"Prior period    : {prior_start} → {prior_end}")
print(f"OSA trend window: {osa_14d_start} → {period_end}")

In [ ]:
# ── Cell 3: Store Validation ──────────────────────────────────────────
store_info_df = spark.sql(f"""
    SELECT DISTINCT store_id, store_name, store_type, reg_id, reg_name
    FROM retail.vw_sales_summary
    WHERE store_id = {store_id}
    LIMIT 1
""")

if store_info_df.count() == 0:
    all_stores = spark.sql("""
        SELECT DISTINCT store_id, store_name FROM retail.vw_sales_summary ORDER BY store_id
    """).toPandas()
    store_list = "\n".join(f"  • {r['store_id']}: {r['store_name']}" for _, r in all_stores.iterrows())
    raise ValueError(
        f"Store ID {store_id} not found. Available stores:\n" + store_list
    )

store_row = store_info_df.first()
STORE_ID   = store_row["store_id"]
STORE_NAME = store_row["store_name"]
STORE_TYPE = store_row["store_type"]
REG_ID     = store_row["reg_id"]
REG_NAME   = store_row["reg_name"]

print(f"✓ Store resolved: {STORE_NAME} (ID: {STORE_ID})")
print(f"  Type: {STORE_TYPE} | Region: {REG_NAME} (ID: {REG_ID})")

---
## Phase 2 — Data Extraction

In [ ]:
# ── Cell 4: Sales Data ─────────────────────────────────────────────────

# Current period sales for target store
sales_current = spark.sql(f"""
    SELECT *
    FROM retail.vw_sales_summary
    WHERE store_id = {STORE_ID}
      AND order_date BETWEEN '{period_start}' AND '{period_end}'
""").toPandas()

# Prior period sales for target store (WoW comparison)
sales_prior = spark.sql(f"""
    SELECT *
    FROM retail.vw_sales_summary
    WHERE store_id = {STORE_ID}
      AND order_date BETWEEN '{prior_start}' AND '{prior_end}'
""").toPandas()

# Region peer sales for benchmarking (current period, same store type)
region_peers = spark.sql(f"""
    SELECT store_id, store_name,
           SUM(line_total)      AS revenue,
           SUM(line_margin_amt) AS margin,
           COUNT(DISTINCT ord_id) AS orders,
           SUM(qty)             AS units
    FROM retail.vw_sales_summary
    WHERE reg_id = {REG_ID}
      AND store_type = '{STORE_TYPE}'
      AND order_date BETWEEN '{period_start}' AND '{period_end}'
    GROUP BY store_id, store_name
    ORDER BY revenue DESC
""").toPandas()

# ── Compute KPIs ──
def compute_sales_kpis(df):
    if df.empty:
        return {"revenue": 0, "orders": 0, "units": 0, "margin": 0,
                "margin_pct": 0, "basket_size": 0}
    revenue  = df["line_total"].sum()
    margin   = df["line_margin_amt"].sum()
    orders   = df["ord_id"].nunique()
    units    = df["qty"].sum()
    return {
        "revenue":    round(revenue, 2),
        "orders":     int(orders),
        "units":      int(units),
        "margin":     round(margin, 2),
        "margin_pct": round(margin / revenue * 100, 1) if revenue else 0,
        "basket_size": round(revenue / orders, 2) if orders else 0,
    }

kpi_current = compute_sales_kpis(sales_current)
kpi_prior   = compute_sales_kpis(sales_prior)

# WoW change
def wow_pct(curr, prev):
    if prev == 0:
        return None
    return round((curr - prev) / prev * 100, 1)

kpi_current["revenue_wow"]  = wow_pct(kpi_current["revenue"], kpi_prior["revenue"])
kpi_current["orders_wow"]   = wow_pct(kpi_current["orders"],  kpi_prior["orders"])
kpi_current["margin_wow"]   = wow_pct(kpi_current["margin"],  kpi_prior["margin"])

# Top/bottom departments
dept_revenue = (
    sales_current.groupby("depa_name")
    .agg(revenue=("line_total", "sum"), margin=("line_margin_amt", "sum"), units=("qty", "sum"))
    .reset_index()
    .sort_values("revenue", ascending=False)
)
dept_prior = (
    sales_prior.groupby("depa_name")
    .agg(revenue_pw=("line_total", "sum"))
    .reset_index()
)
dept_revenue = dept_revenue.merge(dept_prior, on="depa_name", how="left")
dept_revenue["wow_pct"] = dept_revenue.apply(
    lambda r: wow_pct(r["revenue"], r["revenue_pw"]) if pd.notna(r.get("revenue_pw")) else None,
    axis=1,
)

# Daily revenue trend
daily_revenue = (
    sales_current.groupby("order_date")
    .agg(revenue=("line_total", "sum"), orders=("ord_id", "nunique"))
    .reset_index()
    .sort_values("order_date")
)
daily_prior = (
    sales_prior.groupby("order_date")
    .agg(revenue=("line_total", "sum"))
    .reset_index()
    .sort_values("order_date")
)

# Peer rank
if not region_peers.empty:
    peer_rank = int((region_peers["revenue"] >= kpi_current["revenue"]).sum())
    peer_count = len(region_peers)
    peer_avg_revenue = round(region_peers["revenue"].mean(), 2)
else:
    peer_rank = peer_count = 0
    peer_avg_revenue = 0

print(f"Revenue: ${kpi_current['revenue']:,.2f} (WoW: {kpi_current['revenue_wow']}%)")
print(f"Orders:  {kpi_current['orders']}  |  Units: {kpi_current['units']}")
print(f"Margin:  ${kpi_current['margin']:,.2f} ({kpi_current['margin_pct']}%)")
print(f"Basket:  ${kpi_current['basket_size']:,.2f}")
print(f"Peer rank: #{peer_rank} of {peer_count} in {REG_NAME} ({STORE_TYPE})")

In [ ]:
# ── Cell 5: Inventory Data ─────────────────────────────────────────────
# Note: No store-level OSA/OOS data available.
# Inventory health is at warehouse/region level — pulled in Cell 7.

# Placeholder variables for downstream cells (no OSA/OOS views exist)
inv_risk_summary = inv_health["risk_flag"].value_counts().to_dict() if 'inv_health' in dir() else {}
print("Inventory data pulled in Cell 7 (vw_inventory_health).")
print("No store-level shelf availability (OSA) data available.")


In [ ]:
# ── Cell 6: Labor Data ─────────────────────────────────────────────────

# Store labor
labor_df = spark.sql(f"""
    SELECT *
    FROM retail.vw_labor_coverage
    WHERE store_id = {STORE_ID}
""").toPandas()

# Store-type average headcount (benchmark)
type_avg = spark.sql(f"""
    SELECT store_id,
           COUNT(DISTINCT associate_id) AS headcount
    FROM retail.vw_labor_coverage
    WHERE store_type = '{STORE_TYPE}'
    GROUP BY store_id
""").toPandas()

store_type_avg_headcount = round(type_avg["headcount"].mean(), 1) if not type_avg.empty else 0

# ── Compute labor KPIs ──
headcount = labor_df["associate_id"].nunique() if not labor_df.empty else 0

shift_breakdown = (
    labor_df.groupby("shift")
    .agg(
        associates=("associate_id", "nunique"),
        avg_orders=("orders_managed", "mean"),
        avg_tasks=("tasks_completed", "mean"),
    )
    .reset_index()
) if not labor_df.empty else pd.DataFrame()

role_breakdown = (
    labor_df.groupby("role")
    .agg(associates=("associate_id", "nunique"))
    .reset_index()
    .sort_values("associates", ascending=False)
) if not labor_df.empty else pd.DataFrame()

# Single-associate shifts (coverage risk)
single_associate_shifts = (
    shift_breakdown[shift_breakdown["associates"] == 1]["shift"].tolist()
    if not shift_breakdown.empty else []
)

avg_orders_per_assoc = round(labor_df["orders_managed"].mean(), 1) if not labor_df.empty else 0
avg_tasks_per_assoc  = round(labor_df["tasks_completed"].mean(), 1) if not labor_df.empty else 0

is_understaffed = headcount < store_type_avg_headcount

labor_kpis = {
    "headcount": headcount,
    "store_type_avg": store_type_avg_headcount,
    "is_understaffed": is_understaffed,
    "single_associate_shifts": single_associate_shifts,
    "avg_orders_per_assoc": avg_orders_per_assoc,
    "avg_tasks_per_assoc": avg_tasks_per_assoc,
}

print(f"Headcount: {headcount} (type avg: {store_type_avg_headcount})")
print(f"Understaffed: {'YES' if is_understaffed else 'No'}")
if single_associate_shifts:
    print(f"⚠ Single-associate shifts: {', '.join(single_associate_shifts)}")
print(f"Avg orders/associate: {avg_orders_per_assoc}")
print(f"Avg tasks/associate:  {avg_tasks_per_assoc}")

In [ ]:
# ── Cell 7: Inventory Health ───────────────────────────────────────────

inv_health = spark.sql(f"""
    SELECT *
    FROM retail.vw_inventory_health
    WHERE reg_id = {REG_ID}
""").toPandas()

# Perishable lead-time risks
perishable_risks = inv_health[inv_health["risk_flag"] == "PERISHABLE_LEAD_TIME_RISK"]

# Discontinued products still in inventory
discontinued = inv_health[inv_health["risk_flag"] == "DISCONTINUED"]

# Risk flag summary
inv_risk_counts = inv_health["risk_flag"].value_counts().to_dict()

print(f"Inventory records in region {REG_NAME}: {len(inv_health)}")
print(f"Risk flags: {inv_risk_counts}")
if not perishable_risks.empty:
    print(f"\n⚠ Perishable lead-time risks ({len(perishable_risks)} products):")
    for _, r in perishable_risks.head(5).iterrows():
        print(f"  • {r['prod_name']} — lead time {r['lead_time_days']}d vs shelf life {r['shelf_life_days']}d")

---
## Phase 3 — Insight Engine: Risks, Opportunities & Cross-Domain Correlations

In [ ]:
# ── Cell 8: Risk Detection ─────────────────────────────────────────────

risks = []

# ── Sales Risks ──
rev_wow = kpi_current.get("revenue_wow")
if rev_wow is not None and rev_wow <= -REVENUE_CRITICAL_DROP:
    dollar_drop = kpi_prior["revenue"] - kpi_current["revenue"]
    risks.append({
        "severity": "CRITICAL",
        "domain": "Sales",
        "title": f"Revenue declined {abs(rev_wow)}% WoW",
        "detail": f"Revenue dropped from ${kpi_prior['revenue']:,.0f} to ${kpi_current['revenue']:,.0f}.",
        "dollar_impact": round(dollar_drop, 2),
        "reasoning": "A decline exceeding 15% WoW signals a significant disruption — check for staffing gaps or demand shifts.",
    })
elif rev_wow is not None and rev_wow <= -REVENUE_WARNING_DROP:
    dollar_drop = kpi_prior["revenue"] - kpi_current["revenue"]
    risks.append({
        "severity": "WARNING",
        "domain": "Sales",
        "title": f"Revenue declined {abs(rev_wow)}% WoW",
        "detail": f"Revenue dropped from ${kpi_prior['revenue']:,.0f} to ${kpi_current['revenue']:,.0f}.",
        "dollar_impact": round(dollar_drop, 2),
        "reasoning": "A moderate decline warrants investigation into category trends and competitive dynamics.",
    })

# ── Inventory Risks: Perishable lead-time ──
if not perishable_risks.empty:
    risk_products = perishable_risks.head(5)[["prod_name", "lead_time_days", "shelf_life_days"]]
    risks.append({
        "severity": "WARNING",
        "domain": "Inventory",
        "title": f"{len(perishable_risks)} perishable products with lead time > shelf life",
        "detail": "; ".join(
            f"{r['prod_name']} (lead {r['lead_time_days']}d vs shelf {r['shelf_life_days']}d)"
            for _, r in risk_products.iterrows()
        ),
        "dollar_impact": None,
        "reasoning": "When lead time exceeds shelf life, products may arrive already expired. Consider sourcing from closer warehouses or cross-dock facilities.",
    })

# ── Inventory Risks: Discontinued products still in inventory ──
if not discontinued.empty:
    risks.append({
        "severity": "MONITORING",
        "domain": "Inventory",
        "title": f"{len(discontinued)} discontinued products still in regional inventory",
        "detail": "; ".join(discontinued.head(5)["prod_name"].tolist()),
        "dollar_impact": None,
        "reasoning": "Discontinued products occupy warehouse space and tie up capital. Plan markdowns or returns.",
    })

# ── Labor Risks ──
if is_understaffed:
    risks.append({
        "severity": "WARNING",
        "domain": "Labor",
        "title": f"Understaffed: {headcount} associates vs {store_type_avg_headcount} type avg",
        "detail": f"Store has {headcount} associates; {STORE_TYPE} average is {store_type_avg_headcount}.",
        "dollar_impact": None,
        "reasoning": "Understaffed stores show slower shelf replenishment and reduced customer service, directly impacting sales.",
    })

if single_associate_shifts:
    severity = "CRITICAL" if headcount == 0 else "MONITORING"
    risks.append({
        "severity": severity,
        "domain": "Labor",
        "title": f"Single-associate shifts: {', '.join(single_associate_shifts)}",
        "detail": "These shifts have only 1 associate — no coverage for breaks, emergencies, or high-traffic periods.",
        "dollar_impact": None,
        "reasoning": "Single coverage creates operational fragility. Any absence leaves the shift uncovered.",
    })

# Sort by severity
severity_order = {"CRITICAL": 0, "WARNING": 1, "MONITORING": 2}
risks.sort(key=lambda r: severity_order.get(r["severity"], 3))

print(f"Detected {len(risks)} risks:")
for r in risks:
    icon = {"CRITICAL": "🔴", "WARNING": "🟡", "MONITORING": "🟢"}.get(r["severity"], "⚪")
    dollar = f" | ${r['dollar_impact']:,.0f} impact" if r.get("dollar_impact") else ""
    print(f"  {icon} [{r['severity']}] {r['domain']}: {r['title']}{dollar}")


In [ ]:
# ── Cell 9: Opportunity Detection ──────────────────────────────────────

opportunities = []

# ── Opportunity 1: Top-margin categories with peer upside ──
if not dept_revenue.empty and not region_peers.empty:
    # Get region-level department performance
    region_dept = spark.sql(f"""
        SELECT depa_name,
               SUM(line_total) / COUNT(DISTINCT store_id) AS avg_dept_revenue
        FROM retail.vw_sales_summary
        WHERE reg_id = {REG_ID}
          AND store_type = '{STORE_TYPE}'
          AND order_date BETWEEN '{period_start}' AND '{period_end}'
        GROUP BY depa_name
    """).toPandas()

    dept_compare = dept_revenue.merge(region_dept, on="depa_name", how="left")
    dept_compare["gap"] = dept_compare["avg_dept_revenue"] - dept_compare["revenue"]
    underperforming = dept_compare[dept_compare["gap"] > 0].sort_values("gap", ascending=False)

    if not underperforming.empty:
        total_gap = underperforming["gap"].sum()
        top_gaps = underperforming.head(3)
        opportunities.append({
            "domain": "Sales",
            "title": f"Close ${total_gap:,.0f}/week gap vs peer stores in {len(underperforming)} departments",
            "detail": "; ".join(
                f"{r['depa_name']}: ${r['revenue']:,.0f} vs ${r['avg_dept_revenue']:,.0f} peer avg (gap: ${r['gap']:,.0f})"
                for _, r in top_gaps.iterrows()
            ),
            "estimated_dollar_value": round(total_gap, 2),
            "reasoning": "This store is underperforming its peer group in these departments. Investigate assortment, merchandising, and staffing allocation.",
        })

# ── Opportunity 2: Staffing ROI for understaffed high-revenue store ──
if is_understaffed and kpi_current["revenue"] > peer_avg_revenue:
    staff_gap = store_type_avg_headcount - headcount
    opportunities.append({
        "domain": "Labor",
        "title": f"Add {staff_gap:.0f} associates to support above-average revenue",
        "detail": (
            f"This store generates ${kpi_current['revenue']:,.0f}/week (peer avg: ${peer_avg_revenue:,.0f}) "
            f"with only {headcount} associates ({STORE_TYPE} avg: {store_type_avg_headcount}). "
            f"Additional staff could improve service and reduce risk."
        ),
        "estimated_dollar_value": None,
        "reasoning": "High-revenue stores with below-average staffing are at risk of declining performance. Investment in staff protects revenue.",
    })

# ── Opportunity 3: Address perishable lead-time risks ──
if not perishable_risks.empty:
    opportunities.append({
        "domain": "Inventory",
        "title": f"Review sourcing for {len(perishable_risks)} perishable products with lead time > shelf life",
        "detail": (
            "These products risk arriving expired. Consider cross-dock facilities "
            "or closer warehouse sourcing to reduce lead times."
        ),
        "estimated_dollar_value": None,
        "reasoning": "Reducing lead time on perishable goods prevents waste and improves availability.",
    })

print(f"Detected {len(opportunities)} opportunities:")
for o in opportunities:
    dollar = f" | ${o['estimated_dollar_value']:,.0f}" if o.get("estimated_dollar_value") else ""
    print(f"  💡 {o['domain']}: {o['title']}{dollar}")


In [ ]:
# ── Cell 10: Cross-Domain Correlations ─────────────────────────────────

correlations = []

# Pattern 1: High productivity + understaffed → burnout risk
if is_understaffed and avg_orders_per_assoc > 0:
    # Compare to region average productivity
    region_prod = spark.sql(f"""
        SELECT AVG(orders_managed) AS avg_orders
        FROM retail.vw_labor_coverage
        WHERE store_type = '{STORE_TYPE}'
    """).toPandas()
    region_avg_orders = round(region_prod["avg_orders"].iloc[0], 1) if not region_prod.empty else 0

    if avg_orders_per_assoc > region_avg_orders * 1.2:  # 20% above average
        correlations.append({
            "domains": ["Labor"],
            "title": "Team stretched thin — burnout risk",
            "detail": (
                f"Associates manage {avg_orders_per_assoc} orders each (region avg: {region_avg_orders}) "
                f"while the store is understaffed ({headcount} vs {store_type_avg_headcount} avg). "
                f"High workload per person increases turnover and error risk."
            ),
            "reasoning": "Sustained high workload with insufficient headcount leads to burnout, mistakes, and eventual attrition — further worsening the staffing gap.",
        })

# Pattern 2: Revenue decline + understaffed → staffing-driven revenue loss
if rev_wow is not None and rev_wow < -REVENUE_WARNING_DROP and is_understaffed:
    correlations.append({
        "domains": ["Sales", "Labor"],
        "title": "Revenue decline may be linked to understaffing",
        "detail": (
            f"Revenue is down {abs(rev_wow)}% WoW while the store is understaffed "
            f"({headcount} vs {store_type_avg_headcount} avg). Insufficient staff may be "
            f"reducing customer service quality and shelf replenishment speed."
        ),
        "reasoning": "When revenue falls concurrently with low staffing, the drop is often execution-driven. Address the staffing gap before concluding demand has weakened.",
    })

# Pattern 3: Single-associate shifts + revenue decline → shift-specific gap
if single_associate_shifts and rev_wow is not None and rev_wow < -REVENUE_WARNING_DROP:
    correlations.append({
        "domains": ["Labor", "Sales"],
        "title": "Single-coverage shifts may be contributing to revenue decline",
        "detail": (
            f"Shifts with single coverage ({', '.join(single_associate_shifts)}) combined with "
            f"a {abs(rev_wow)}% revenue decline suggests customer service gaps "
            f"during understaffed periods."
        ),
        "reasoning": "Lone associates prioritize immediate tasks over proactive customer engagement. Adding coverage during these shifts can improve sales.",
    })

print(f"Detected {len(correlations)} cross-domain correlations:")
for c in correlations:
    print(f"  🔗 {' × '.join(c['domains'])}: {c['title']}")
    print(f"     {c['detail']}")


---
## Phase 4 — Visualizations

In [ ]:
# ── Cell 11: Sales Dashboard ───────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle(f"Sales Dashboard — {STORE_NAME}", fontsize=14, fontweight="bold")

# ── 11a: Daily Revenue Trend (current period) ──
ax1 = axes[0]
if not daily_revenue.empty:
    ax1.plot(
        daily_revenue["order_date"], daily_revenue["revenue"],
        marker="o", linewidth=2, color="#2563EB", label="Current Period"
    )
    ax1.fill_between(daily_revenue["order_date"], daily_revenue["revenue"], alpha=0.1, color="#2563EB")
ax1.set_title("Daily Revenue Trend")
ax1.set_ylabel("Revenue ($)")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax1.tick_params(axis="x", rotation=45)
ax1.legend()

# ── 11b: Department Revenue (horizontal bar) ──
ax2 = axes[1]
if not dept_revenue.empty:
    top_depts = dept_revenue.head(8)
    colors = ["#EF4444" if (w is not None and w < 0) else "#22C55E"
              for w in top_depts["wow_pct"]]
    ax2.barh(top_depts["depa_name"], top_depts["revenue"], color=colors)
    for i, (_, row) in enumerate(top_depts.iterrows()):
        wow_label = f" ({row['wow_pct']:+.0f}%)" if row["wow_pct"] is not None else ""
        ax2.text(row["revenue"], i, f" ${row['revenue']:,.0f}{wow_label}", va="center", fontsize=8)
ax2.set_title("Revenue by Department")
ax2.set_xlabel("Revenue ($)")
ax2.invert_yaxis()

# ── 11c: Margin % by Category ──
ax3 = axes[2]
if not dept_revenue.empty:
    dept_revenue["margin_pct"] = (dept_revenue["margin"] / dept_revenue["revenue"] * 100).round(1)
    top_margin = dept_revenue.sort_values("margin_pct", ascending=False).head(8)
    ax3.barh(top_margin["depa_name"], top_margin["margin_pct"], color="#8B5CF6")
    for i, (_, row) in enumerate(top_margin.iterrows()):
        ax3.text(row["margin_pct"], i, f" {row['margin_pct']:.1f}%", va="center", fontsize=8)
ax3.set_title("Margin % by Department")
ax3.set_xlabel("Margin (%)")
ax3.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 12: Inventory Dashboard ───────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f"Inventory Health — {STORE_NAME} (Region: {REG_NAME})", fontsize=14, fontweight="bold")

# ── 12a: Risk Flag Distribution (Donut) ──
ax1 = axes[0]
if inv_risk_counts:
    labels = list(inv_risk_counts.keys())
    sizes  = list(inv_risk_counts.values())
    colors_map = {
        "PERISHABLE_LEAD_TIME_RISK": "#F59E0B",
        "DISCONTINUED": "#EF4444",
        "OK": "#22C55E"
    }
    colors = [colors_map.get(l, "#94A3B8") for l in labels]
    wedges, texts, autotexts = ax1.pie(
        sizes, labels=labels, colors=colors, autopct="%1.0f%%",
        pctdistance=0.8, startangle=90,
        wedgeprops={"width": 0.4, "edgecolor": "white"}
    )
    for t in autotexts:
        t.set_fontsize(8)
else:
    ax1.text(0.5, 0.5, "No inventory data", ha="center", va="center")
ax1.set_title("Inventory Risk Flags")

# ── 12b: Top Perishable Lead-Time Risks ──
ax2 = axes[1]
if not perishable_risks.empty:
    top_perish = perishable_risks.head(8).copy()
    top_perish["days_over"] = top_perish["lead_time_days"] - top_perish["shelf_life_days"]
    ax2.barh(top_perish["prod_name"], top_perish["days_over"], color="#F59E0B")
    for i, (_, row) in enumerate(top_perish.iterrows()):
        ax2.text(row["days_over"], i,
                 f" lead {row['lead_time_days']}d vs shelf {row['shelf_life_days']}d",
                 va="center", fontsize=7)
    ax2.set_xlabel("Days Over Shelf Life")
    ax2.invert_yaxis()
else:
    ax2.text(0.5, 0.5, "No perishable lead-time risks", ha="center", va="center")
ax2.set_title("Perishable Products: Lead Time > Shelf Life")

plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 13: Labor Dashboard ───────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle(f"Labor Coverage — {STORE_NAME}", fontsize=14, fontweight="bold")

# ── 13a: Shift Coverage ──
ax1 = axes[0]
if not shift_breakdown.empty:
    bars = ax1.bar(shift_breakdown["shift"], shift_breakdown["associates"], color="#2563EB")
    # Highlight single-associate shifts
    for bar, (_, row) in zip(bars, shift_breakdown.iterrows()):
        if row["associates"] == 1:
            bar.set_color("#EF4444")
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 str(int(row["associates"])), ha="center", va="bottom", fontweight="bold")
    ax1.axhline(y=1, color="#EF4444", linestyle="--", alpha=0.5, label="Single coverage")
    ax1.legend(fontsize=8)
ax1.set_title("Associates per Shift")
ax1.set_ylabel("Headcount")

# ── 13b: Productivity per Associate ──
ax2 = axes[1]
if not labor_df.empty:
    ax2.scatter(labor_df["orders_managed"], labor_df["tasks_completed"],
                alpha=0.7, c="#8B5CF6", edgecolors="white", s=60)
    ax2.axvline(x=avg_orders_per_assoc, color="#94A3B8", linestyle="--", alpha=0.5, label=f"Avg orders ({avg_orders_per_assoc})")
    ax2.axhline(y=avg_tasks_per_assoc, color="#94A3B8", linestyle=":", alpha=0.5, label=f"Avg tasks ({avg_tasks_per_assoc})")
    ax2.legend(fontsize=8)
ax2.set_title("Associate Productivity")
ax2.set_xlabel("Orders Managed")
ax2.set_ylabel("Tasks Completed")

# ── 13c: Store vs Type Average ──
ax3 = axes[2]
categories = ["Headcount", "Avg Orders/Assoc", "Avg Tasks/Assoc"]
store_vals  = [headcount, avg_orders_per_assoc, avg_tasks_per_assoc]
# Get type averages for comparison
type_avg_prod = spark.sql(f"""
    SELECT AVG(orders_managed) AS avg_orders,
           AVG(tasks_completed) AS avg_tasks
    FROM retail.vw_labor_coverage
    WHERE store_type = '{STORE_TYPE}'
""").toPandas()
type_avg_orders = round(type_avg_prod["avg_orders"].iloc[0], 1) if not type_avg_prod.empty else 0
type_avg_tasks  = round(type_avg_prod["avg_tasks"].iloc[0], 1) if not type_avg_prod.empty else 0
type_vals = [store_type_avg_headcount, type_avg_orders, type_avg_tasks]

x = range(len(categories))
width = 0.35
ax3.bar([i - width/2 for i in x], store_vals, width, label=STORE_NAME, color="#2563EB")
ax3.bar([i + width/2 for i in x], type_vals, width, label=f"{STORE_TYPE} Avg", color="#94A3B8")
ax3.set_xticks(list(x))
ax3.set_xticklabels(categories)
ax3.legend()
ax3.set_title(f"Store vs {STORE_TYPE} Average")

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 14: Risk & Opportunity Summary Visual ─────────────────────────

# ── 14a: Risk Matrix ──
if risks:
    risk_df = pd.DataFrame(risks)
    risk_summary = risk_df.groupby("severity").size().reindex(
        ["CRITICAL", "WARNING", "MONITORING"], fill_value=0
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f"Risk & Opportunity Summary — {STORE_NAME}", fontsize=14, fontweight="bold")

    ax1 = axes[0]
    colors = ["#DC2626", "#F59E0B", "#22C55E"]
    bars = ax1.bar(risk_summary.index, risk_summary.values, color=colors)
    for bar, val in zip(bars, risk_summary.values):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 str(val), ha="center", va="bottom", fontweight="bold", fontsize=14)
    ax1.set_title("Risks by Severity")
    ax1.set_ylabel("Count")

    # ── 14b: Opportunity Dollar Values ──
    ax2 = axes[1]
    opp_with_value = [o for o in opportunities if o.get("estimated_dollar_value")]
    if opp_with_value:
        opp_labels = [o["title"][:40] + "..." if len(o["title"]) > 40 else o["title"]
                      for o in opp_with_value]
        opp_values = [o["estimated_dollar_value"] for o in opp_with_value]
        domain_colors = {"Inventory": "#2563EB", "Sales": "#22C55E", "Labor": "#8B5CF6"}
        opp_colors = [domain_colors.get(o["domain"], "#94A3B8") for o in opp_with_value]
        ax2.barh(opp_labels, opp_values, color=opp_colors)
        for i, v in enumerate(opp_values):
            ax2.text(v, i, f" ${v:,.0f}/wk", va="center", fontsize=9)
        ax2.invert_yaxis()
    ax2.set_title("Opportunities (Est. Weekly $ Value)")
    ax2.set_xlabel("Estimated Value ($)")

    plt.tight_layout()
    plt.show()
else:
    print("No risks detected — store is performing well across all dimensions.")

---
## Phase 5 — LLM Narrative Synthesis

In [ ]:
# ── Cell 15: Azure OpenAI Reasoning ────────────────────────────────────
# Uses Azure OpenAI to generate an executive narrative from computed insights.
# Configure your endpoint and key below, or set via Fabric Key Vault / env vars.

try:
    from openai import AzureOpenAI

    # ── Configuration — update these or use environment variables ──
    import os
    AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "<your-endpoint>")
    AZURE_OPENAI_KEY      = os.environ.get("AZURE_OPENAI_KEY", "<your-key>")
    AZURE_OPENAI_MODEL    = os.environ.get("AZURE_OPENAI_MODEL", "gpt-4o")

    client = AzureOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_KEY,
        api_version="2024-12-01-preview",
    )

    # Package all insights into a structured prompt
    insight_payload = {
        "store": {"name": STORE_NAME, "id": STORE_ID, "type": STORE_TYPE, "region": REG_NAME},
        "period": f"{period_start} to {period_end}",
        "sales_kpis": kpi_current,
        "peer_context": {"rank": peer_rank, "total_peers": peer_count, "peer_avg_revenue": peer_avg_revenue},
        "labor": labor_kpis,
        "risks": risks,
        "opportunities": opportunities,
        "cross_domain_correlations": correlations,
        "top_departments": dept_revenue.head(5).to_dict(orient="records") if not dept_revenue.empty else [],
    }

    system_prompt = """You are a retail operations analyst generating an executive store intelligence report.

RULES:
- Reference ONLY the data provided. Never invent numbers, product names, or store names.
- Include specific dollar values where available.
- Explain WHY each risk or opportunity matters (root cause reasoning).
- Be direct and actionable — this is for busy store leaders.

Generate the following sections:

## Executive Summary
3–5 sentences: What's happening at this store and why. Lead with the most important finding.

## Key Risks
For each risk, explain:
- What is happening
- Why it matters (root cause hypothesis)
- Dollar impact if available
Group by severity (Critical → Warning → Monitoring).

## Opportunities
For each opportunity:
- What action to take
- Expected dollar impact
- Why this works (reasoning)

## Cross-Domain Insights
Explain how issues in different domains (sales, inventory, labor) are connected.

## Recommended Actions
Top 3–5 prioritized, specific actions. Each should name a specific product, shift, or department.
"""

    user_prompt = f"""Generate a store intelligence report for the following data:

{json.dumps(insight_payload, indent=2, default=str)}
"""

    response = client.chat.completions.create(
        model=AZURE_OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.3,
        max_tokens=2000,
    )

    llm_narrative = response.choices[0].message.content
    print("✓ LLM narrative generated successfully.")

except Exception as e:
    print(f"⚠ LLM synthesis skipped: {e}")
    print("  Falling back to rule-based summary.")

    # ── Fallback: rule-based narrative ──
    sections = []
    sections.append(f"## Executive Summary\n")
    sections.append(
        f"{STORE_NAME} generated ${kpi_current['revenue']:,.0f} in revenue over the past {lookback_days} days "
        f"({'↑' if (rev_wow or 0) >= 0 else '↓'}{abs(rev_wow or 0)}% WoW). "
        
        f"The store has {headcount} associates ({'below' if is_understaffed else 'at/above'} the {STORE_TYPE} average of {store_type_avg_headcount}). "
        f"{len(risks)} risks and {len(opportunities)} opportunities were identified.\n"
    )

    if risks:
        sections.append("## Key Risks\n")
        for r in risks:
            icon = {"CRITICAL": "🔴", "WARNING": "🟡", "MONITORING": "🟢"}.get(r["severity"], "")
            dollar = f" — **${r['dollar_impact']:,.0f} impact**" if r.get("dollar_impact") else ""
            sections.append(f"### {icon} {r['severity']}: {r['title']}{dollar}\n")
            sections.append(f"{r['detail']}\n")
            sections.append(f"*Reasoning: {r['reasoning']}*\n")

    if opportunities:
        sections.append("## Opportunities\n")
        for o in opportunities:
            dollar = f" — **${o['estimated_dollar_value']:,.0f}/week**" if o.get("estimated_dollar_value") else ""
            sections.append(f"### 💡 {o['title']}{dollar}\n")
            sections.append(f"{o['detail']}\n")
            sections.append(f"*Reasoning: {o['reasoning']}*\n")

    if correlations:
        sections.append("## Cross-Domain Insights\n")
        for c in correlations:
            sections.append(f"### 🔗 {c['title']}\n")
            sections.append(f"{c['detail']}\n")
            sections.append(f"*{c['reasoning']}*\n")

    sections.append("## Recommended Actions\n")
    action_num = 1
    for r in risks:
        if r["severity"] == "CRITICAL" and action_num <= 5:
            sections.append(f"{action_num}. **{r['domain']}**: Address {r['title']}\n")
            action_num += 1
    for o in opportunities:
        if action_num <= 5:
            sections.append(f"{action_num}. **{o['domain']}**: {o['title']}\n")
            action_num += 1

    llm_narrative = "\n".join(sections)

In [ ]:
# ── Cell 16: Final Report Assembly ─────────────────────────────────────

from datetime import datetime as dt

# ── Key Metrics summary table ──
metrics_html = f"""
<table style="width:100%; border-collapse:collapse; margin:16px 0;">
<tr style="background:#1E40AF; color:white; font-weight:bold;">
  <td style="padding:8px;">Revenue</td>
  <td style="padding:8px;">WoW %</td>
  <td style="padding:8px;">Orders</td>
  <td style="padding:8px;">Margin %</td>
  <td style="padding:8px;">Basket Size</td>
  <td style="padding:8px;">Headcount</td>
  <td style="padding:8px;">Peer Rank</td>
</tr>
<tr style="font-size:18px; font-weight:bold;">
  <td style="padding:8px;">${kpi_current['revenue']:,.0f}</td>
  <td style="padding:8px; color:{'#22C55E' if (rev_wow or 0) >= 0 else '#EF4444'};">
    {'+' if (rev_wow or 0) >= 0 else ''}{rev_wow or 'N/A'}%</td>
  <td style="padding:8px;">{kpi_current['orders']}</td>
  <td style="padding:8px;">{kpi_current['margin_pct']}%</td>
  <td style="padding:8px;">${kpi_current['basket_size']:,.2f}</td>

  <td style="padding:8px; color:{'#EF4444' if is_understaffed else '#22C55E'};">
    {headcount} / {store_type_avg_headcount} avg</td>
  <td style="padding:8px;">#{peer_rank} of {peer_count}</td>
</tr>
</table>
"""

# ── Risk matrix HTML ──
risk_rows = ""
for r in risks:
    color = {"CRITICAL": "#FEE2E2", "WARNING": "#FEF3C7", "MONITORING": "#DCFCE7"}.get(r["severity"], "#F1F5F9")
    icon  = {"CRITICAL": "🔴", "WARNING": "🟡", "MONITORING": "🟢"}.get(r["severity"], "")
    dollar = f"${r['dollar_impact']:,.0f}" if r.get("dollar_impact") else "—"
    risk_rows += f"""
    <tr style="background:{color};">
      <td style="padding:6px;">{icon} {r['severity']}</td>
      <td style="padding:6px;">{r['domain']}</td>
      <td style="padding:6px;">{r['title']}</td>
      <td style="padding:6px;">{dollar}</td>
    </tr>"""

risk_html = f"""
<h3>Risk Matrix</h3>
<table style="width:100%; border-collapse:collapse;">
<tr style="background:#374151; color:white;">
  <th style="padding:6px; text-align:left;">Severity</th>
  <th style="padding:6px; text-align:left;">Domain</th>
  <th style="padding:6px; text-align:left;">Risk</th>
  <th style="padding:6px; text-align:left;">$ Impact</th>
</tr>
{risk_rows}
</table>
"""

# ── Opportunity table HTML ──
opp_rows = ""
for o in opportunities:
    dollar = f"${o['estimated_dollar_value']:,.0f}/wk" if o.get("estimated_dollar_value") else "—"
    opp_rows += f"""
    <tr>
      <td style="padding:6px;">{o['domain']}</td>
      <td style="padding:6px;">{o['title']}</td>
      <td style="padding:6px; font-weight:bold; color:#16A34A;">{dollar}</td>
    </tr>"""

opp_html = f"""
<h3>Opportunities</h3>
<table style="width:100%; border-collapse:collapse;">
<tr style="background:#374151; color:white;">
  <th style="padding:6px; text-align:left;">Domain</th>
  <th style="padding:6px; text-align:left;">Opportunity</th>
  <th style="padding:6px; text-align:left;">Est. Value</th>
</tr>
{opp_rows}
</table>
"""

# ── Assemble full report ──
report_html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width:1000px; margin:auto;">

<h1 style="border-bottom:3px solid #1E40AF; padding-bottom:8px;">
  Store Intelligence Report: {STORE_NAME}
</h1>
<p style="color:#64748B;">Region: {REG_NAME} | Type: {STORE_TYPE} | Period: {period_start} → {period_end}</p>

<h3>Key Metrics</h3>
{metrics_html}

<hr/>
{risk_html}

<hr/>
{opp_html}

<hr/>
<div style="line-height:1.7;">
{llm_narrative}
</div>

<hr/>
<p style="color:#94A3B8; font-size:12px;">
  Generated {dt.now().strftime('%Y-%m-%d %H:%M:%S')} | Data source: lh_retaildata | 
  Prepared for District & Store Managers
</p>
</div>
"""

displayHTML(report_html)

---
## Phase 7 — Persist Recommendations to Table

Writes all insights (risks, opportunities, correlations, executive summary) to `retail.recommendations` for the Recommendation Data Agent.

In [ ]:
# ── Cell 17: Persist Recommendations to Lakehouse Table ──────────────
# This cell writes all generated insights to retail.recommendations
# so the Recommendation Data Agent can serve them to end users.

from pyspark.sql.types import StructType, StructField, StringType, FloatType
from datetime import datetime as dt

report_date = today
generated_at = dt.now()

# ── Build rows from risks, opportunities, correlations, and summary ──
rows = []

# Risks
for r in risks:
    rows.append({
        "report_date": str(report_date),
        "str_id": str(STORE_ID),
        "str_name": STR_NAME,
        "str_type": STR_TYPE,
        "reg_id": REG_ID,
        "reg_name": REG_NAME,
        "insight_type": "RISK",
        "severity": r["severity"],
        "domain": r["domain"],
        "title": r["title"],
        "detail": r["detail"],
        "reasoning": r["reasoning"],
        "dollar_impact": float(r["dollar_impact"]) if r.get("dollar_impact") else None,
        "generated_at": str(generated_at),
    })

# Opportunities
for o in opportunities:
    rows.append({
        "report_date": str(report_date),
        "str_id": str(STORE_ID),
        "str_name": STR_NAME,
        "str_type": STR_TYPE,
        "reg_id": REG_ID,
        "reg_name": REG_NAME,
        "insight_type": "OPPORTUNITY",
        "severity": None,
        "domain": o["domain"],
        "title": o["title"],
        "detail": o["detail"],
        "reasoning": o["reasoning"],
        "dollar_impact": float(o["estimated_dollar_value"]) if o.get("estimated_dollar_value") else None,
        "generated_at": str(generated_at),
    })

# Cross-domain correlations
for c in correlations:
    rows.append({
        "report_date": str(report_date),
        "str_id": str(STORE_ID),
        "str_name": STR_NAME,
        "str_type": STR_TYPE,
        "reg_id": REG_ID,
        "reg_name": REG_NAME,
        "insight_type": "CORRELATION",
        "severity": None,
        "domain": " × ".join(c["domains"]),
        "title": c["title"],
        "detail": c["detail"],
        "reasoning": c["reasoning"],
        "dollar_impact": None,
        "generated_at": str(generated_at),
    })

# Executive summary (LLM narrative or fallback)
rows.append({
    "report_date": str(report_date),
    "str_id": str(STORE_ID),
    "str_name": STR_NAME,
    "str_type": STR_TYPE,
    "reg_id": REG_ID,
        "reg_name": REG_NAME,
    "insight_type": "EXECUTIVE_SUMMARY",
    "severity": None,
    "domain": "All",
    "title": f"Daily Intelligence Report — {STR_NAME}",
    "detail": llm_narrative,
    "reasoning": None,
    "dollar_impact": None,
    "generated_at": str(generated_at),
})

# ── KPI snapshot row ──
kpi_json = json.dumps({
    "revenue": kpi_current["revenue"],
    "revenue_wow_pct": kpi_current.get("revenue_wow"),
    "orders": kpi_current["orders"],
    "units": kpi_current["units"],
    "margin_pct": kpi_current["margin_pct"],
    "basket_size": kpi_current["basket_size"],
    "headcount": headcount,
    "peer_rank": peer_rank,
    "peer_count": peer_count,
})
rows.append({
    "report_date": str(report_date),
    "str_id": str(STORE_ID),
    "str_name": STR_NAME,
    "str_type": STR_TYPE,
    "reg_id": REG_ID,
        "reg_name": REG_NAME,
    "insight_type": "KPI_SNAPSHOT",
    "severity": None,
    "domain": "All",
    "title": f"KPI Snapshot — {STR_NAME} — {report_date}",
    "detail": kpi_json,
    "reasoning": None,
    "dollar_impact": None,
    "generated_at": str(generated_at),
})

# ── Write to Lakehouse table ──
schema = StructType([
    StructField("report_date", StringType(), False),
    StructField("str_id", StringType(), True),
    StructField("str_name", StringType(), True),
    StructField("str_type", StringType(), True),
    StructField("reg_id", StringType(), True),
    StructField("reg_name", StringType(), True),
    StructField("insight_type", StringType(), False),
    StructField("severity", StringType(), True),
    StructField("domain", StringType(), True),
    StructField("title", StringType(), False),
    StructField("detail", StringType(), True),
    StructField("reasoning", StringType(), True),
    StructField("dollar_impact", FloatType(), True),
    StructField("generated_at", StringType(), False),
])

df_recommendations = spark.createDataFrame(rows, schema=schema)

# Delete existing rows for this store + date (idempotent re-runs)
spark.sql(f"""
    DELETE FROM retail.recommendations
    WHERE str_id = '{STORE_ID}' AND report_date = '{report_date}'
""")

# Append new rows
df_recommendations.write.mode("append").saveAsTable("retail.recommendations")

print(f"✓ Wrote {len(rows)} recommendation rows to retail.recommendations")
print(f"  Store: {STR_NAME} (str_id: {STORE_ID}) | Date: {report_date}")
print(f"  Breakdown: {sum(1 for r in rows if r['insight_type']=='RISK')} risks, "
      f"{sum(1 for r in rows if r['insight_type']=='OPPORTUNITY')} opportunities, "
      f"{sum(1 for r in rows if r['insight_type']=='CORRELATION')} correlations, "
      f"1 executive summary, 1 KPI snapshot")